# ⚠️ Week 3 — Customer Churn Prediction
**Smart E-Commerce Analytics Platform**

**Dataset:** E-Commerce Customer Churn Dataset (Kaggle) → `churn_clean.csv`
- 3,941 customers with behavioral features
- Target: `Churn` (0 = Active, 1 = Churned)

**This notebook covers:**
1. Load & inspect churn dataset
2. Data cleaning & encoding
3. Exploratory analysis of churn drivers
4. Train Random Forest classifier
5. Evaluate: Accuracy, F1, ROC-AUC, Confusion Matrix
6. Feature importance analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              confusion_matrix, roc_curve, classification_report)
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14
print('Libraries loaded ✅')

## 1. Load & Inspect Churn Dataset

In [ ]:
# Load raw churn data
raw = pd.read_csv('../data/churn/data_ecommerce_customer_churn.csv')
print(f'Raw shape: {raw.shape}')
print(f'Columns: {list(raw.columns)}')
display(raw.head(3))
print('\nMissing values:')
print(raw.isnull().sum())
print(f'\nChurn rate (raw): {raw["Churn"].mean():.1%}')

## 2. Data Cleaning

In [ ]:
# Load pre-cleaned version
df = pd.read_csv('../data/churn_clean.csv')
print(f'Cleaned shape: {df.shape}')
print(f'Missing values after cleaning: {df.isnull().sum().sum()}')
print(f'Churn rate: {df["Churn"].mean():.1%}')
print(f'\nClass distribution:')
print(df['Churn'].value_counts())
display(df.describe().round(2))

In [ ]:
# Encode categorical features
le_cat = LabelEncoder()
le_mar = LabelEncoder()
df['PreferedOrderCat_enc'] = le_cat.fit_transform(df['PreferedOrderCat'])
df['MaritalStatus_enc']    = le_mar.fit_transform(df['MaritalStatus'])

print('Category encoding:')
for i, cls in enumerate(le_cat.classes_):
    print(f'  {i} → {cls}')
print('\nMarital status encoding:')
for i, cls in enumerate(le_mar.classes_):
    print(f'  {i} → {cls}')

## 3. Exploratory Analysis of Churn Drivers

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Tenure
axes[0,0].hist(df[df['Churn']==0]['Tenure'], bins=20, alpha=0.6, color='steelblue', label='Active')
axes[0,0].hist(df[df['Churn']==1]['Tenure'], bins=20, alpha=0.6, color='red', label='Churned')
axes[0,0].set_title('Tenure Distribution')
axes[0,0].set_xlabel('Tenure (months)')
axes[0,0].legend()

# Satisfaction Score
churn_by_sat = df.groupby('SatisfactionScore')['Churn'].mean()
axes[0,1].bar(churn_by_sat.index, churn_by_sat.values,
              color=['#2ecc71' if v < 0.2 else '#e67e22' if v < 0.4 else '#e74c3c'
                     for v in churn_by_sat.values])
axes[0,1].set_title('Churn Rate by Satisfaction Score')
axes[0,1].set_xlabel('Score (1=Low, 5=High)')
axes[0,1].set_ylabel('Churn Rate')

# Preferred Category
churn_by_cat = df.groupby('PreferedOrderCat')['Churn'].mean().sort_values(ascending=False)
churn_by_cat.plot(kind='bar', ax=axes[0,2], color='steelblue', edgecolor='white')
axes[0,2].set_title('Churn Rate by Preferred Category')
axes[0,2].set_ylabel('Churn Rate')
axes[0,2].tick_params(axis='x', rotation=30)

# Complain
churn_by_comp = df.groupby('Complain')['Churn'].mean()
axes[1,0].bar(['No Complaint', 'Complained'], churn_by_comp.values,
              color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[1,0].set_title('Churn Rate by Complaint Status')
axes[1,0].set_ylabel('Churn Rate')

# Days Since Last Order
axes[1,1].hist(df[df['Churn']==0]['DaySinceLastOrder'], bins=20, alpha=0.6, color='steelblue', label='Active')
axes[1,1].hist(df[df['Churn']==1]['DaySinceLastOrder'], bins=20, alpha=0.6, color='red', label='Churned')
axes[1,1].set_title('Days Since Last Order')
axes[1,1].set_xlabel('Days')
axes[1,1].legend()

# Cashback
axes[1,2].hist(df[df['Churn']==0]['CashbackAmount'], bins=20, alpha=0.6, color='steelblue', label='Active')
axes[1,2].hist(df[df['Churn']==1]['CashbackAmount'], bins=20, alpha=0.6, color='red', label='Churned')
axes[1,2].set_title('Cashback Amount Distribution')
axes[1,2].set_xlabel('Cashback ($)')
axes[1,2].legend()

plt.suptitle('Churn Analysis by Key Features', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('../report/churn_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Train-Test Split & Model Training

In [ ]:
FEAT_COLS = ['Tenure', 'WarehouseToHome', 'NumberOfDeviceRegistered',
             'SatisfactionScore', 'NumberOfAddress', 'Complain',
             'DaySinceLastOrder', 'CashbackAmount',
             'PreferedOrderCat_enc', 'MaritalStatus_enc']

X = df[FEAT_COLS]
y = df['Churn']

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train: {len(X_tr)} | Test: {len(X_te)}')
print(f'Train churn rate: {y_tr.mean():.1%}')
print(f'Test churn rate:  {y_te.mean():.1%}')

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)
print('\nRandom Forest trained ✅')

# Cross-validation
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='roc_auc')
print(f'5-Fold CV ROC-AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

## 5. Model Evaluation

In [ ]:
y_pred = rf.predict(X_te)
y_prob = rf.predict_proba(X_te)[:, 1]

print('=== MODEL METRICS ===')
print(f'Accuracy : {accuracy_score(y_te, y_pred):.4f}')
print(f'F1 Score : {f1_score(y_te, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_te, y_prob):.4f}')
print()
print(classification_report(y_te, y_pred, target_names=['Active', 'Churned']))

In [ ]:
cm = confusion_matrix(y_te, y_pred)
fpr, tpr, _ = roc_curve(y_te, y_prob)
auc = roc_auc_score(y_te, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Active','Churned'], yticklabels=['Active','Churned'])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

axes[1].plot(fpr, tpr, color='steelblue', linewidth=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0,1],[0,1],'k--', linewidth=1)
axes[1].set_title('ROC Curve — Random Forest')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.savefig('../report/churn_model_eval.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEAT_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
importances.plot(kind='bar', ax=ax, color=sns.color_palette('Blues_r', len(importances)))
ax.set_title('Random Forest Feature Importance')
ax.set_ylabel('Importance')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../report/churn_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 3 churn predictors:')
for feat, imp in importances.head(3).items():
    print(f'  {feat}: {imp:.3f}')

print('\n✅ Churn Prediction Complete!')